In [1]:
!pip install kagglehub pandas

  Obtaining dependency information for kagglehub from https://files.pythonhosted.org/packages/61/1a/6b1293d0b091905e367f86771a7122e5f7f2729ab9f0f36a1617f7ad6c05/kagglehub-1.0.2-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.7 MB/s eta 0:00:00
  Obtaining dependency information for kagglesdk<1.0,>=0.1.22 from https://files.pythonhosted.org/packages/60/85/eedcc1c5f3d198d64dcc6f68c30739abb28c180b889099d4e47bb11581ac/kagglesdk-0.1.30-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 5.6 MB/s eta 0:00:0000:01


# Rain in Australia

## Motivation and Question

### Real-world motivation

Rainfall is one of the most difficult weather events. According to ABC News Australia (Australian Broadcasting Corporation), citing data from the Australia Bureau of Meteorology, Australia's official national weather agency, rainfall is in fact more difficult to forecast than temperature because it has much greater local and temporal variability. Despite major advances in the forecasting system, it remains a challenging tasks in many parts of the world. 

Improving rainfall prediction can helper farmers plan irrigation and harvesting, as well as enable emergency services to respond more effectively to severe weather conditions. It could also help individuals and businesses make better decisions about outdoor activities and operations. 

### Prediction Question

Can we accurately predict whether it will rain tomorrow in Australia using today’s weather metrics?

## Target Variable and Class Labels

### Target Variable 
RainTomorrow

### Class labels 
* Yes - It will rain tomorrow
* No - It will not rain tomorrow

This is a binary classification problem

## Data Source, Observations, and Main Features

### Dataset source
* Kaggle: Rain in Australia (Weather Dataset Rattle Package)

### Dataset overview

* 145,000 daily weather observations
* 23 variables
* Weather records collected from multiple Australian locations over about 10 years

### Main Features

* Date & Location
    * Date
    * Location
* Temperature
    * MinTemp
    * MaxTemp
    * Temp9am
    * Temp3pm
* Rainfall
    * Rainfall
    * RainToday
* Wind
    * WindGustSpeed
    * WindGustDir
    * WindSpeed9am
    * WindSpeed3pm
    * WindDir9am
    * WindDir3pm
* Atmospheric Conditions
    * Humidity9am
    * Humidity3pm
    * Pressure9am
    * Pressure3pm
    * Cloud9am
    * Cloud3pm
    * Sunshine
    * Evaporation
* Target Variable
    * RainTomorrow (Yes / No)

### Import Dataset

In [2]:
import kagglehub
from pathlib import Path

# Download latest version
handle = "jsphyg/weather-dataset-rattle-package"
path = "weatherAUS.csv"
if not Path(path).exists():
    print("Downloading from Kaggle")
    path = kagglehub.dataset_download(
        "jsphyg/weather-dataset-rattle-package",
        "weatherAUS.csv",
        output_dir="."
    )

print("Path to dataset files:", path)

Path to dataset files: weatherAUS.csv


In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv(path)

## Cleaning and Preprocessing

* Handling null values should happend before enrichment.
    * Except some enrichment should happend before getting rid of null values. For example, consecutive rain days
* Enrichment should happen before EDA
* EDA should happen before preprocessing
* Some amount of EDA should handle any imputed (estimated and added in) null values, if we're using any
    * E.g. Box plot with and without imputed values

* Handle categorical values ([ordinal][sklearn.preprocessing.OrdinalEncoder], [one-hot][sklearn.preprocessing.OneHotEncoder], [target][sklearn.preprocessing.TargetEncoder])
* [Standardize scalers][sklearn.preprocessing.StandardScaler]?
* Handle null values
    * Delete: simply drop any rows/columns with missing values, especially if they are few and missingness is random
    * Impute: replace with estimated values
        * with mean, median, or mode
        * using forward/backward fill - duplicating in the previous/next seen value...
    * Interpolate from nearby values...

[sklearn.preprocessing.OrdinalEncoder]: https://scikit-learn.org/1.9/modules/generated/sklearn.preprocessing.OrdinalEncoder.html#sklearn.preprocessing.OrdinalEncoder
[sklearn.preprocessing.OneHotEncoder]: https://scikit-learn.org/1.9/modules/generated/sklearn.preprocessing.OneHotEncoder.html#sklearn.preprocessing.OneHotEncoder
[sklearn.preprocessing.TargetEncoder]: https://scikit-learn.org/1.9/modules/generated/sklearn.preprocessing.TargetEncoder.html#sklearn.preprocessing.TargetEncoder
[sklearn.preprocessing.StandardScaler]: https://scikit-learn.org/1.9/modules/generated/sklearn.preprocessing.StandardScaler.html

According to the [Australian Government's Bureau of Meterology](https://www.bom.gov.au/climate/dwo/IDCJDW0000.shtml), "From time to time, observations will not be available, for a variety of reasons. Sometimes when the daily maximum and minimum temperatures, rainfall or evaporation are missing, the next value given has been accumulated over several days rather than the normal one day. It is very difficult for an automatic system to detect this reliably, so caution is advised."

In [9]:
print(f"Number of columns with a significant amount of missing data: {((df.isnull().sum() / df.shape[0]) > 0.05).sum()} / {df.shape[1]} = {((df.isnull().sum() / df.shape[0]) > 0.05).sum() / df.shape[1]}")
print(f"Number of rows with at least one missing feature: {(df.isnull().sum(axis=1) > 0).sum()} / {df.shape[0]} = {(df.isnull().sum(axis=1) > 0).sum() / df.shape[0]}")
print(f"Number of rows with more than one missing feature: {(df.isnull().sum(axis=1) > 2).sum()} / {df.shape[0]} = {(df.isnull().sum(axis=1) > 2).sum() / df.shape[0]}")

Number of columns with a significant amount of missing data: 9 / 23 = 0.391304347826087
Number of rows with at least one missing feature: 89040 / 145460 = 0.6121270452358036
Number of rows with more than one missing feature: 59785 / 145460 = 0.41100646225766535


In [10]:
from sklearn.impute import SimpleImputer
# 1. Remove rows with missing target variable
df = df[df["RainTomorrow"].notna()].copy()

In [11]:
# Make sure Date is datetime
df["Date"] = pd.to_datetime(df["Date"])

# Create season/month
df["Month"] = df["Date"].dt.month

# Separate columns
target = "RainTomorrow"
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
categorical_cols.remove(target)
numerical_cols = df.select_dtypes(include=["number"]).columns.tolist()

In [12]:
# Numerical: fill by Location + Month median first
for col in numerical_cols:
    df[col] = df.groupby(["Location", "Month"])[col].transform(
        lambda x: x.fillna(x.median())
    )
    # fallback if some Location+Month group is still all missing
    df[col] = df[col].fillna(df[col].median())

# Categorical: fill by Location mode first
for col in categorical_cols:
    df[col] = df.groupby("Location")[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
    )
    # fallback
    df[col] = df[col].fillna(df[col].mode()[0])

/var/folders/0q/k242vzws4fl4_f2tkzxdrqym0000gn/T/ipykernel_25450/3730477369.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)


Instead of using a single global median for all missing numerical values, we used grouped imputation based on location and month. This is more appropriate for weather data because temperature, rainfall, humidity, and pressure vary strongly across regions and seasons. For categorical variables, missing values were filled using the most frequent value within each location, with a global mode as a fallback. Rows with missing RainTomorrow were removed because the target label should not be artificially created.

In [ ]:
# Encode target
df[target] = df[target].map({"No": 0, "Yes": 1})
print("Missing values remaining:", df.isnull().sum().sum())

Missing values remaining: 0


### Enrich Dataset
- TempRange — MaxTemp minus MinTemp
- MonsoonSeason — location-aware wet/monsoon flag (Yes/No)
- WindSpeedDiff — WindSpeed3pm minus WindSpeed9am (signed)
- HumidityDiff — Humidity3pm minus Humidity9am (signed)
- TempDiff9am3pm — Temp3pm minus Temp9am (signed)
- PressureDiff — Pressure3pm minus Pressure9am (signed)
- DewPoint9am — dew point from 9am temp and humidity
- Season — Southern Hemisphere season from date
- DaysSinceRain — days since last rain per location
- ConsecutiveRainDays — consecutive rainy days before current day
- LaNina — monthly La Niña flag from NOAA NINO3.4 anomaly

In [5]:
from enrich_weather import enrich
df = enrich(df)

In [7]:
print("Number of columns with a significant amount of missing data", ((df.isnull().sum() / df.shape[0]) > 0.05).sum())

Number of columns with a significant amount of missing data 10


## Exploratory Data Analysis
e.g. Description: columns, data-types, null values, counts, mean, standard deviation, box plot: min, 25%, 50%, 75% max; Distribution: target, target vs. key features, correlation matrix; target class balance

## Train-Test Split

## Simple Model

## Interpretable Model
Logistic regression or decision tree

## Improved Model
Random forest, boosting, other ensemble

## Evaluation
With confucion matrix and suitable classification matrix

## Limitations

### False Positives and Negatives

## References